In [3]:
import numpy as np
import random
from tqdm.auto import tqdm

In [68]:
x = [
    [1, 2, 3]
]

y = [3,4,5]

# convert normal python array into numpy ndarray
x = np.array(x)

PROBLEM_SIZE  = np.shape(x)[0]
PROBLEM_SIZE

1

In [65]:
# print(np.divide(5,2))
# print(np.remainder(5,2))
# print(np.pow(5,2))

# a x b = b x 1  ->  a x 1
# 3 x 3 * 3 x 
# y = np.add(np.array(x[0]), 5)
print(y)

# print(np.array(x).shape)

[3, 4, 5]


In [50]:
binary_operators = [np.add, np.subtract, np.dot, np.divide, np.pow]

BINARY_OPERATORS = {
    "+": np.add,
    "-": np.subtract,
    "*": np.dot,
    "/": np.divide,
    "^": np.pow
}

#https://numpy.org/doc/2.1/reference/routines.math.html
UNARY_OPERATORS = {
        "": lambda x: x,  
        "sin": np.sin,
        "cos": np.cos,
        "tan":np.tan,
        "log": np.log,
        "exp": np.exp,
        "arccos": np.arccos,
        "arcsin":np.arcsin,
        "arctan":np.arctan,
        "sqrt":np.sqrt,
        "cbrt":np.cbrt,
        "square":np.square,
        "abs":np.abs,
        "reciprocal":np.reciprocal
    }

#VARIABLES = [f"X_{i}" for i in range(PROBLEM_SIZE)]

#VARIABLES_WEIGHTS = [[1/len(VARIABLES) for _ in range(len(VARIABLES))]]
VARIABLES_MAP = {f"X_{i}": x[i] for i in range(PROBLEM_SIZE)}    # {'X_0': [1, 2, 3], 'X_1': [4, 5, 6], 'X_2': [7, 8, 9]}
print(VARIABLES_MAP)

{'X_0': array([1, 2, 3])}


## Tree structure

In [87]:
class TreeNode:
    def __init__(self, value):
        self.value = value  # This can be an operator or operand
        self.left = None    # Left child
        self.right = None   # Right child

def validate_tree(node):
    if not node:
        return True
    
    if node.value in BINARY_OPERATORS:
        if not node.left or not node.right:
            return False  # Operators must have two children
        return validate_tree(node.left) and validate_tree(node.right)
    
    elif node.value in UNARY_OPERATORS:  # Allow unary operators
        if not node.left and node.right:
            return False  # Unary operators must have one child on the left
        return validate_tree(node.left)
    
    # elif node.value in VARIABLES_MAP and isinstance(node.value, str):  # Allow variables
    elif node.value in VARIABLES_MAP:
        return True
    else:
        return False  # Invalid value
    
    
# (3 + 2) * (4 + 5)        Treenode (value = *, left = Treenode (value = +, left = 3, right = 2), right = Treenode (value = +, left = 4, right = 5))
def evaluate_tree(node):
    if not node:
        return 0
    if node.value in BINARY_OPERATORS:
        left_val = evaluate_tree(node.left)
        right_val = evaluate_tree(node.right)
        return BINARY_OPERATORS[node.value](left_val, right_val)
    
    elif node.value in UNARY_OPERATORS:
        left_val = evaluate_tree(node.left)
        return UNARY_OPERATORS[node.value(left_val)] 
    
    elif node.value in VARIABLES_MAP:  # Variable
        return VARIABLES_MAP[node.value]  # Replace with variable value
    
    else:  # Coefficient
        return node.value
    
def random_tree(depth):
    if depth == 0:
        # Leaf node (operand or variable)
        variable = np.random.choice(list(VARIABLES_MAP.keys()))
        values = np.array([i for i in range(10)])
        values.append(variable)
        return TreeNode(np.random.choice(values))
    elif np.random.rand() > 0.5:
        # Unary operator
        node = TreeNode(np.random.choice(list(UNARY_OPERATORS.keys())))
        node.left = random_tree(depth - 1)
        return node
    else:
        # Binary operator
        node = TreeNode(np.random.choice(list(BINARY_OPERATORS.keys())))
        node.left = random_tree(depth - 1)
        node.right = random_tree(depth - 1)
        return node


### printing functions

In [ ]:
def print_tree(node):
    if not node:
        return
    print_tree(node.left)
    print(node.value, end=" ")
    print_tree(node.right)

def print_expr(node):
    print_tree(node) 
    print("= y")

def print_tree_values(node):
    if not node:
        return
    print_tree_values(node.left)
    print(VARIABLES_MAP[node.value] if node.value in VARIABLES_MAP else node.value, end=" ")
    print_tree_values(node.right)

def print_expr_values(node):
    print_tree_values(node) 
    print(" = ", end="")
    print(evaluate_tree(node))

In [88]:
tree = random_tree(3)
tree

ValueError: a must be 1-dimensional or an integer

In [78]:
root = TreeNode("+")
root.left = TreeNode(3)
root.right = TreeNode("*")
root.right.left = TreeNode(4)
root.right.right = TreeNode("X_0") # 3 + 4x = 3 + 4 x[1]


print_expr(root)
print("--------------------------")
print_expr_values(root)

3 + 4 * X_0  = y
--------------------------
3 + 4 * [1 2 3]  = [ 7 11 15]


In [157]:
def mse(x,y):
    return (x-y)**2

In [158]:
initial_solution = TreeNode(operators[random.randint(0,num_operators)])
print(initial_solution.value)
initial_solution.left = TreeNode(x[0])
initial_solution.right = TreeNode(random.randint(1,10))
print(initial_solution.right.value)
print(evaluate_tree(initial_solution,{}))

# y = x+ n
# mse(evaluate_tree(initial_solution, {}), y[0])
# tree
#   operator
#       |
#      / \
#    x    n

for i in range(200):
    sol = TreeNode(initial_solution.value)
    sol.left = initial_solution.left
                                
    # mutation
    sol.value = operators[random.randint(0,num_operators)]
    sol.right =  TreeNode(random.randint(1,10))

    ev = evaluate_tree(sol, {})
    # print(ev)
    if mse(evaluate_tree(initial_solution,{}), y[0]) > mse(ev,y[0]):
        initial_solution.value = sol.value
        initial_solution.right = sol.right
        print("found better solution")
        print(mse(evaluate_tree(sol, {}),y))


print(evaluate_tree(initial_solution,{}))
print(f"{initial_solution.left.value} {initial_solution.value} {initial_solution.right.value}")
    



<ufunc 'subtract'>
2
0
found better solution
[7.36734694]
found better solution
[1]
found better solution
[0]
3
2 <ufunc 'add'> 1
